# Lithuanian speech-to-text — Parakeet (Kmynas) on Colab

Transcribes Lithuanian audio to a punctuated, speaker-labelled transcript
plus `.srt` / `.vtt` / `.json`, using a NeMo Parakeet-TDT fine-tune.

Faster than the Whisper pipeline (`colab_quickstart.ipynb`) — real-time
factor around 0.03 on a T4, so a 35-minute recording decodes in about a
minute — and the model punctuates by itself.

**Set the runtime to GPU**: Runtime → Change runtime type → T4 GPU.
CPU works but is roughly 30× slower.

Repo: https://github.com/kristijonasatpro/paprika


## 1. Install

NeMo is a large install — budget 5–10 minutes. It pins versions that
conflict with the Whisper pipeline, which is why that one lives in a
separate environment locally; on Colab just don't run both notebooks in
the same session.


In [ ]:
!git clone -q https://github.com/kristijonasatpro/paprika
%cd paprika
!pip install -q 'nemo_toolkit[asr]' silero-vad soundfile sherpa-onnx

import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '— good to go')
else:
    print('NO GPU. Runtime > Change runtime type > T4 GPU, then rerun.')


## 2. Get the model

Point `MODEL` at any `.nemo` Parakeet checkpoint. If it lives in a private
Hugging Face repo, add an `HF_TOKEN` in Colab secrets (key icon, left
sidebar) first — a free read-only token is enough.


In [ ]:
import os
from huggingface_hub import hf_hub_download

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded')
except Exception:
    print('No HF_TOKEN secret — fine for public repos, needed for private ones.')

REPO = 'kristijonas/kmynas-parakeet-lt-v3'   # change me
FILE = 'kmynas-parakeet-lt-v3.nemo'          # change me
MODEL = hf_hub_download(REPO, FILE)
print('model at', MODEL)


## 3. Add your audio

Run the cell and pick a file (m4a, mp3, wav, mp4, mov — anything ffmpeg
reads), or mount Drive instead:

```python
from google.colab import drive; drive.mount('/content/drive')
AUDIO = '/content/drive/MyDrive/your-recording.m4a'
```


In [ ]:
from google.colab import files
up = files.upload()
AUDIO = list(up)[0]
print('using', AUDIO)


## 4. Transcribe

`--speakers N` when you know how many people are talking — it is much more
reliable than letting the clustering threshold guess. `--no-diar` skips
speaker labels and is faster.

On CUDA the default `fp16` is fine. (On Apple Silicon it is not always —
use `--dtype fp32` there.)


In [ ]:
!python transcribe_kmynas.py "$AUDIO" \
    --model "$MODEL" \
    --speakers 2 \
    --lexicon lexicon.tsv


## 5. Read it, then download


In [ ]:
import pathlib
stem = pathlib.Path(AUDIO).with_suffix('')
print(pathlib.Path(f'{stem}.txt').read_text()[:4000])


In [ ]:
for ext in ('txt', 'srt', 'vtt', 'json'):
    p = pathlib.Path(f'{stem}.{ext}')
    if p.exists():
        files.download(str(p))


## Tuning, if the output looks wrong

| symptom | flag to reach for |
|---|---|
| short consonant-cluster junk (`chl`, `žl`) on breath or laughter | raise `--min-confidence` toward `0.99` |
| real quiet words disappearing | lower `--min-confidence`, or `0` to disable |
| words broken across block joins | you are probably in `--no-vad` mode; drop the flag |
| out of GPU memory | lower `--vad-target` (60 → 30) |
| the same borrowed word spelled several ways | edit `lexicon.tsv` |
| speaker labels obviously wrong | pass `--speakers N`, or `--no-diar` |

Block length is the memory knob and attention cost grows with its square.
A T4's 15 GB handles 60 s blocks comfortably. Whole-file single-pass
decoding (no blocks) works up to roughly 5 minutes of audio and runs out
of memory well before 10.

The design reasoning behind each of these — including several fixes that
were tried and measured *worse* — is in the docstrings of
`transcribe_kmynas.py` and `chunk_longform.py`.
